# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified relaxation buoyancy field and surface wind stress.

twnh Dec '25

This notebook solves

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code derives the equation satisfied by the surface pressure field $p_s(x,y)$.
It then solves for the pressure field given a specific simple example of specified windstress and simple buoyancy forcing.

The buoyancy forcing comes from solving

\begin{align}
 \epsilon^2   \kappa \frac{d^2 b}{d z^2} + \gamma ( B(z) - b ) = 0
\end{align}
for $b(z)$, given diffusivity $\kappa = \kappa_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= 0 ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field and $\gamma$ is the relaxation rate.

This version builds on `ExampleSolution_v_0.4.ipynb` by replacing `SymPy.lambdify` calls with `Symbolics.build_function` calls, hopefully to speed up the slow final calculation. The code still includes a lot of `SymPy` and attempts (not 100% successfully) to convert to `Symbolics` at the end. This code also includes several checks to verify explicit analytical forms (in the LaTeX notes, `PlanetaryGeostrophySolutions_v0.5.tex`) match the SymPy calculations.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
notebook_name = "ExampleSolution_v0.5"
using Infiltrator
using Symbolics
const I = im    # Define I as the imaginary unit
using Gridap
using GridapGmsh
using GridapMakie
using Gridap.Geometry
using Gridap.ReferenceFEs
using WriteVTK
using Profile
using ProfileView

#### Function to convert a `SymPy` expression to `Symbolics`. This replaces `lambdify` calls (slow). However, for some complicated `SymPy` expressions, it's too slow to make this conversion.

In [ ]:
function build_Symbolics_fn(expr,varnames)
	vars = [Symbolics.Variable(Symbol(n)) for n in varnames]							# Extract Symbolics variables from variable names
	tmp1 = sympy_to_symbolics(expr, vars)			
	I_vars = filter(v -> string(v) == "I", Symbolics.get_variables(tmp1))				# Check if imaginary unit variable is present and substitute
	if !isempty(I_vars)
    	tmp2 = substitute(tmp1, Dict(first(I_vars) => im))
	else
	    tmp2 = tmp1
	end
	tmp3vec = build_function(tmp2, vars; expression=Val{false})

	# Wrap function to throw an error if it's called with multiple scalar arguments rather than a vector:
	function tmp3(args::AbstractVector)
    	length(args) == length(vars) || throw(ArgumentError("Input must be vector of length $(length(vars))."))
    	tmp3vec(args)
	end
	tmp3(args...) = throw(MethodError(tmp3, args))

	return tmp3
end

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x,y)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τs     = SymFunction("τs",    complex=true)         # Complex surface wind stress
psg    = SymFunction("psg",   complex=true)         # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)
b      = SymFunction("b",     real=true)            # Buoyancy field b(x,y,z)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν) 

# Buoyancy equation symbolic parameters:
κ₀, ψ  = symbols("κ₀ ψ",  real=true, positive=true) # Diffusivity parameters
γ, α   = symbols("γ α",   real=true, positive=true) # Relaxation parameter
B      = SymFunction("B", real=true)                # Relaxation buoyancy field

# Define diffusivity profile here:
κ = κ₀                                              # Constant viscosity profile

b_params = (κ, ϵ, γ) 

# Convenience variables:
root_iϕ = ϕ*sympy.sqrt(im)
sample_params = Dict(ν₀=>1//2, κ₀=>1, ϵ=>2, ψ=>3, root_iϕ=>4im, ϕ => 4, H(x,y)=>1 - x^2 - y^2)
sample_point  = Dict(x=>-0.6, y=>0.2, z=>-0.32)

### Compute pressure fields for the specified buoyancy field $b$:

In [ ]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient

### Function to solve the frictional geostrophic equation using a Green's function:

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                                  # Unknown coefficient in the Green's function solution

    # 0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert sympy.simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert sympy.simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert sympy.simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert sympy.simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final sympy.simplify (to cancel constants). Avoid sympy.simplify in general because it's not always reproducible.
    G = sympy.simplify(G)
    return G
end ;

### Compute the velocity G's function:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)

# Used for stress-driven part of the velocity:
Guv0 = Guv.subs(ξ,0)
Latex_Guv0 = (im * sympy.sqrt(im) * exp(-root_iϕ * z)/(ν₀ * ϵ^2 * ϕ * (1 + exp(2 * root_iϕ * H(x,y))))) *
(exp(2 * root_iϕ * (z + H(x,y))) - 1)
resid = Guv0 - Latex_Guv0
@assert resid == 0
println("0.1 LaTeX Guv0 expression matches.")

# Used for surface-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_int_wrt_ξ = (im / (ν₀ * ϵ^2 * ϕ^2 * (1 + exp(2 * root_iϕ * H(x,y)))) ) *
(1 + exp(2 * root_iϕ * H(x,y)) - exp(root_iϕ * (z + H(x,y))) - exp(root_iϕ * (H(x,y) - z)))
resid = Guv_int_wrt_ξ - Latex_Guv_int_wrt_ξ
@assert resid == 0
println("0.2 LaTeX Guv integral wrt ξ expression matches.")

# Used for baroclinic-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv * exp(ψ * ξ)),(ξ,-H(x,y),0))
Guv_exp_psi_xi_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_exp_psi_xi_int_wrt_ξ = (1/(ν₀ * ϵ^2 * (ψ^2 - im * ϕ^2))) * (
    ( im * sympy.sqrt(im) * ψ * exp(      ψ * H(x,y)) * (exp(2 * root_iϕ * (z + H(x,y))) - 1) -
                            ϕ * exp(root_iϕ * H(x,y)) * (exp(2 * root_iϕ *  z          ) + 1)
    ) * exp(-root_iϕ * z - ψ * H(x,y))
    /(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))
    + exp(ψ * z)
)
resid = sympy.expand(Guv_exp_psi_xi_int_wrt_ξ - Latex_Guv_exp_psi_xi_int_wrt_ξ)
@assert resid == 0
println("0.3 LaTeX Guv * exp(ψ * z) integral matches.")

#### Function to solve the buoyancy equation using a Green's function:

In [ ]:
function compute_Gb(b_params, geometry_params)
    # Setup symbols and parameters:
    κ, ϵ, γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    
    #0. Define the ODE for b(z):
    ode = Eq( - γ * b(z)     + ϵ^2 * diff(diff(κ *  b(z),z),z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bm)) == 0                  # Check solution
    @assert sympy.simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bp)) == 0                  # Check solution
    @assert sympy.simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = sympy.simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; ξ)$:
    Gm = sympy.simplify(bm * bp.subs(z,ξ) / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))
    Gp = sympy.simplify(bm.subs(z,ξ) * bp / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * κ.subs(z,ξ)) == 0

    # Define piecewise Green's function:
    G = sympy.simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert sympy.simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert sympy.simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

### Compute the buoyancy Green's function:

In [ ]:
Gb_sym  = compute_Gb(b_params,geometry_params)
Gb = Gb_sym.subs(γ,ψ^2 * ϵ^2 * κ₀)

#### Compute the buoyancy term explicitly for a linear relaxation buoyancy profile. This is to check and support the Latex derivation in `PlanetaryGeostrophySolutions_V0.5.tex`. The symbolic expressions themselves are used in `ExampleSolution_v0.6.ipynb`, which avoid SymPy by using `Symbolics.jl` instead. `Symbolics.jl` can't manage the integrations done by `Sympy` in this notebook.

In [ ]:
# Compute $z$ integral of Gb for Latex notes:
Gb_int = SymPy.integrate(Gb,(z,z,0))
Gb_int = Gb_int.subs(max_obj, z)
Gb_int = sympy.simplify(Gb_int)
Latex_Gb_int_den = 2 * κ₀ * ψ^2 * ϵ^2 * (exp(2 * ψ * H(x,y)) + 1)
Latex_Gb_int_num = 
    exp(   ψ * ( z - ξ + 2 * H(x,y))) -
    exp(   ψ * ( z + ξ + 2 * H(x,y))) +
2 * exp(   ψ * ( ξ + 2 * H(x,y) ) ) + 
2 * exp( - ψ * ξ) +  
    exp(   ψ * ( ξ - z )) - 
    exp( - ψ * ( z + ξ )) -
    exp(   ψ * ( ξ + 2 * H(x,y) -  sympy.Max( z, ξ ))) - 
    exp( - ψ * ( ξ - 2 * H(x,y) -  sympy.Max( z, ξ ) )) -
    exp(   ψ * ( ξ - sympy.Max( z, ξ ) )) - 
    exp( - ψ * ( ξ - sympy.Max( z, ξ ) )) 

@assert sympy.simplify(Latex_Gb_int_num / Latex_Gb_int_den - Gb_int) == 0
println("1.1 LaTeX Gb integral matches.")

# Compute integrand for linear relaxation buoyancy profile explicitly:
tmp_integrand = sympy.expand(- Gb_int * ξ * α * γ)
tmp_integral  = sympy.integrate(tmp_integrand, (ξ, -H(x,y), 0))
tmp_integral  = sympy.simplify(tmp_integral.subs(max_obj, z))
Latex_Gb_int_int = z^2 +
    2*( 2 * exp(ψ * H(x,y)) - exp(ψ * (z + H(x,y))) - exp(ψ * (H(x,y) - z)) ) / (ψ^2 * (exp(2 * ψ * H(x,y)) + 1) )
Latex_Gb_int_int = (- α * γ) * Latex_Gb_int_int / (2 * κ₀ * ϵ^2 * ψ^2)
@assert sympy.simplify(Latex_Gb_int_int  - tmp_integral) == 0
println("1.2 LaTeX Gb double integral with linear buoyancy relaxation profile matches.")

# Differentiate Latex_Gb_int_int to get the baroclinic pressure gradient:
dGb_int_int_∂z = sympy.simplify(diff(Latex_Gb_int_int,x) + im*diff(Latex_Gb_int_int,y))
Latex_dGb_int_int_∂z = 
( ( (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y))) * (exp(ψ * z) - 2 + exp(- ψ * z)) ) / ( exp(2 * ψ * H(x,y)) + 1 )^2 ) * (diff(H(x,y),x) + im * diff(H(x,y),y))
Latex_dGb_int_int_∂z = (α * γ) * Latex_dGb_int_int_∂z / (κ₀ * ϵ^2 * ψ^3)
@assert sympy.simplify(dGb_int_int_∂z - Latex_dGb_int_int_∂z) == 0
println("1.3 LaTeX vertical integral of horizontal buoyancy derivative with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related T(x) term (excludes the stress-driven part):
Txb_term_integrand = exp(-root_iϕ * (ξ - H(x,y))) * (1 + exp(2*root_iϕ * ξ)) / (1 + exp(2*root_iϕ * H(x,y)))
Txb_term_integrand = Txb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Txb_term = sympy.integrate(sympy.expand(Txb_term_integrand), (ξ, -H(x,y), 0))
Txb_term = sum(Txb_term.args[i] for i in 1:length(Txb_term.args)-1) + Txb_term.args[length(Txb_term.args)].args[1].args[1]    # Manually expand the integral result

Latex_Txb_term = α * γ * (exp((ψ + root_iϕ) * H(x,y)) - exp((3*ψ + root_iϕ) * H(x,y)))/
(κ₀ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2 * (1 + exp(2 * root_iϕ * H(x,y)) ))
Latex_Txb_term = Latex_Txb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) 
)
# Check intermediate analytic Latex integral:
inter1 = exp(- (ψ - root_iϕ)*ξ) + exp(- (ψ + root_iϕ)*ξ) - 2*(exp(-root_iϕ*ξ) + exp(+root_iϕ*ξ)) + exp((ψ + root_iϕ)*ξ) + exp((ψ - root_iϕ)*ξ)
inter2 = sympy.integrate(inter1, (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Txb_term = Latex_Txb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Txb_term) - sympy.expand(Txb_term)).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.4 LaTeX T(x) buoyancy term with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related B(x) term (excludes the stress-driven part):
Bxb_term_integrand = exp(-root_iϕ * ξ) * (exp(root_iϕ * ξ) - exp(root_iϕ * H(x,y))) * (exp(root_iϕ * (ξ + H(x,y))) - 1)
Bxb_term_integrand = Bxb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Bxb_term = sympy.integrate(sympy.expand(Bxb_term_integrand), (ξ, -H(x,y), 0))

Latex_Bxb_term = α * γ * (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y)))/(κ₀ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2)
Latex_Bxb_term = Latex_Bxb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = exp(root_iϕ * H(x,y)) * (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) +
((exp((-ψ + root_iϕ) * H(x,y)) - exp((ψ + root_iϕ) * H(x,y)))/ψ)
)
Latex_int_result = Latex_int_result + (1/ψ)*(exp(-ψ * H(x,y)) - exp(ψ * H(x,y))) + 2*H(x,y)*(1 + exp(2*root_iϕ * H(x,y)))
# Check intermediate analytic Latex integral:
inter1 = (1 - exp(root_iϕ * (H(x,y) - ξ))) * (exp(root_iϕ * (H(x,y) + ξ)) - 1) * (exp(ψ * ξ) - 2 + exp(- ψ * ξ))
inter2 = sympy.integrate(sympy.expand(inter1), (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Bxb_term = Latex_Bxb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Bxb_term) - Bxb_term).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.5 LaTeX B(x) buoyancy term with linear buoyancy relaxation profile matches.")

### Symbolic computation of buoyancy $b$

In [ ]:
# Compute buoyancy field b(z):
b_sym = SymPy.integrate(sympy.expand(-Gb * γ * B(x,y,ξ)), (ξ,-H(x,y),0) )
b = sympy.simplify(b_sym.subs(max_obj,z))

# Check that the final expression for b satisfies the original differential equation:
tmp1 = ϵ^2 * diff(diff(κ * b,z),z) - γ * b
tmp1 = sympy.simplify(tmp1.subs(ψ,sqrt(γ/κ₀)/ϵ))
@assert tmp1 + γ * B(x,y,z) == 0

### Symbolic computation of flow $(u(z),v(z)), U, V, \tau_b$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(sympy.expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
𝔲1 = 𝔲1.subs(max_obj,z)
𝔲2 = - Guv0 * τs(x,y)
𝔲 = 𝔲1 + 𝔲2

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + ϵ^2 * diff(diff(ν * 𝔲1,z),z)
tmp1 = sympy.simplify(tmp1.subs(ϕ,sqrt(f/ν₀)/ϵ))
tmp2 = -im * f * 𝔲2 + ϵ^2 * diff(diff(ν * 𝔲2,z),z)
tmp2 = sympy.simplify(tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ))
@assert tmp1 + tmp2 == pg

# Compute depth-integrated flow:
𝔘 = SymPy.integrate(sympy.expand(𝔲),(z,-H(x,y),0))

# Compute bottom stress on fluid:
τb = - ϵ^2 * ν * diff(𝔲,z).subs(z,-H(x,y))

# Check expression for surface stress on fluid:
@assert sympy.simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ϵ^2 * ν * diff(𝔲,z).subs(z,0)
@assert sympy.simplify(tmp - τs(x,y)) == 0

### Check final results from LaTeX derivation:

In [ ]:
# Check integral of surface pressure term for bottom stress:
tmp_τ_integrand = sympy.expand( (1 + exp(2*root_iϕ*ξ)) * exp(-root_iϕ*(ξ-H(x,y))) / (1 + exp(2*root_iϕ*H(x,y))) )
tmp = sympy.simplify(sympy.integrate(tmp_τ_integrand,(ξ,-H(x,y),0)).args[1].args[1])
Latex_tmp = (1/root_iϕ) * sympy.tanh(root_iϕ*H(x,y))
@assert tmp == Latex_tmp
println("2.1 LaTeX integral of the τb surface pressure term matches.")

# Check S(x) function:
Latex_S_fn = Latex_tmp
XXX = symbols("XXX")                            # SymPy can't pull the constant psg factor outside the integral.
τb_tmp = sympy.expand(τb.subs(psg(x,y),XXX).doit())
psg_coeff = τb_tmp.coeff(XXX)
testS = sympy.simplify(Latex_S_fn - sympy.simplify(psg_coeff))
@assert testS == 0
println("2.2 LaTeX S(x) function matches.")

# Check T(x) function:
function make_Latex_T_fn(int_bg_term)
    tmp_T_integrand2 = sympy.expand(tmp_τ_integrand * int_bg_term)
    tmp_T_fn = ( sympy.integrate(tmp_T_integrand2,(ξ,-H(x,y),0)) - τs(x,y)*(2*exp(root_iϕ*H(x,y)))/(1 + exp(2*root_iϕ*H(x,y))))
    return tmp_T_fn
end
int_bg_term = sympy.integrate(bg(x,y,ξ),(ξ,ξ,0))        # Generic integral of bg.
Latex_T_fn = make_Latex_T_fn(int_bg_term)

τb_tmp2 = (τb_tmp - psg_coeff * XXX).doit()
testT = τb_tmp2 - Latex_T_fn
YYY = symbols("YYY")                            # SymPy can't work on the buried bg integral, so substitute for it
testT = testT.subs(bg(x,y,ξ),YYY)
testT = sympy.expand(testT).doit()
@assert testT == 0
println("2.3 LaTeX T(x) function matches.")

# Check integral of surface pressure term for 𝔘:
tmp_ps_integrand = sympy.expand(exp(-root_iϕ*ξ)*(exp(root_iϕ*ξ) - exp(root_iϕ*H(x,y)))*(exp(root_iϕ*(ξ + H(x,y))) - 1))
tmp = sympy.integrate(tmp_ps_integrand,(ξ,-H(x,y),0))
Latex_tmp = - H(x,y) * (exp(2*root_iϕ*H(x,y)) + 1) + (1/root_iϕ)*(exp(2*root_iϕ*H(x,y)) - 1)
@assert tmp == Latex_tmp
println("2.4 LaTeX integral of the 𝔘 surface pressure term matches.")

# Check A(x) function:
Latex_A_fn = (im/f)*(H(x,y) + (1/root_iϕ)*((1 - exp(2*root_iϕ*H(x,y))) / (1 + exp(2*root_iϕ*H(x,y)) ) )).subs(ϕ,sqrt(f/ν₀)/ϵ)
𝔘_tmp = sympy.expand(𝔘.subs(psg(x,y),XXX).doit())
psg_coeff = 𝔘_tmp.coeff(XXX).subs(ϕ,sqrt(f/ν₀)/ϵ)
testA = sympy.simplify(Latex_A_fn - psg_coeff)
@assert testA == 0
println("2.5 LaTeX A(x) function matches.")

# Check B(x) function:
function make_Latex_B_fn(int_bg_term)
    tmp_B_integrand2 = sympy.expand(tmp_ps_integrand * int_bg_term)
    tmp_B_fn = (-im/(f*(1+exp(2*root_iϕ*H(x,y))))) * ( sympy.integrate(tmp_B_integrand2,(ξ,-H(x,y),0)) + τs(x,y)*(exp(root_iϕ*H(x,y)) - 1)^2)
    return tmp_B_fn
end
Latex_B_fn = make_Latex_B_fn(int_bg_term)
𝔘_tmp2 = (𝔘_tmp - psg_coeff * XXX).doit()
testB = (𝔘_tmp2 - Latex_B_fn).subs(ϕ,sqrt(f/ν₀)/ϵ)
testB = testB.subs(bg(x,y,ξ),YYY)
testB = sympy.expand(testB).doit()
@assert testB == 0
println("2.6 LaTeX B(x) function matches.")

### Define parameter values:

In [ ]:
# Define the parameter values
f_val  = 1
ϵ_val  = 0.95
ν₀_val = 0.06
κ₀_val = ν₀_val
γ_val  = 0.2

# Compute compound parameter:
ϕ_val = sqrt(f_val / ν₀_val) / ϵ_val
ψ_val = sqrt(γ_val / κ₀_val) / ϵ_val

# Define domain depth:
H_val(x,y)    = 1.0 - x^2 - y^2

# Define wind stress:
# τs_val(x,y) = 0.1 + 0.0im
function τs_val(x, y)
    r = sqrt(x^2 + y^2)
    V = 0.0 * r^2           # Define this function for your use-case
    return -V * y / r + im* V * x / r
end

# Define relaxation buoyancy field:
B_val(x,y,ξ)  = α * ξ
α_val = 0.3
B_valH(x, y)  = B_val(x,y,-H_val(x,y) )

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val, ϵ=>ϵ_val, ν₀=>ν₀_val, ϕ=>ϕ_val, γ=>γ_val, κ₀=>κ₀_val, ψ=>ψ_val, α=>α_val)
fn_values    = Dict(
    τs(x,y)=>τs_val(x,y), 
    H(x, y)=>H_val(x,y)
    )
B_values     = Dict(
    B(x, y, ξ)=>B_val(x, y, ξ), 
    B(x, y, z)=>B_val(x, y, z), 
    B(x, y, -H(x,y))=>B_valH(x,y)
    )

#### Compute buoyancy expression for Latex notes and check it:

In [ ]:
Latex_b = ((- α * γ)) / (κ₀ * ψ^2 * ϵ^2) * ( 
(exp(ψ * H(x,y)) * (exp(ψ * z) - exp(-ψ * z))) /(ψ * (1 + exp(2 * ψ * H(x,y))))
- z
)
@assert b.subs(B_values) - Latex_b == 0
println("2.7 LaTeX b(x) buoyancy expression with linear buoyancy relaxation profile matches.")

#### Compute baroclinic pressure gradient term: This is demanding!

In [ ]:
@time begin
    println()
    println("Computing baroclinic pressure gradient term...")
    this_b = sympy.simplify(b.subs(B_values).doit())
    this_pbarog = sympy.simplify(diff(this_b,x) + im*diff(this_b,y))
    this_pbarog_intz = sympy.simplify(sympy.integrate(sympy.expand(this_pbarog),(z,ξ,0)))         # This isn't too crazy!
    B_values[SymPy.integrate(bg(x,y,ξ),(ξ,ξ,0))] = this_pbarog_intz

    Latex_T_fn = make_Latex_T_fn(this_pbarog_intz)
    Latex_T_fn = Latex_T_fn.subs(param_values).subs(B_values).subs(fn_values).doit()
    Latex_B_fn = make_Latex_B_fn(this_pbarog_intz)
    Latex_B_fn = Latex_B_fn.subs(param_values).subs(B_values).subs(fn_values).doit()

    println("done.")
end

#### Report:

In [ ]:
println()
println("This case parameter values:")
display(param_values)

println()
println("This case function values:")
display(fn_values)

println()
println("This relaxation buoyancy function values:")
display(B_values)

println()
Ekman_depth = sqrt(2*ν₀_val/f_val)
println("Non-dimensional Ekman_depth = $(Ekman_depth)")

### Solve for surface pressure using Gridap

In [ ]:
println()
println("Solving for the surface pressure using Gridap...")

@time begin

# 1. Define the mesh
    model = GmshDiscreteModel("unit_circle_v0.3.msh")
    Ω = Triangulation(model)
    dΩ = Measure(Ω, 2)

# 2. Define the finite element space (piecewise linear,Dirichlet zero BC)
    order = 2
    reffe = ReferenceFE(lagrangian, Float64, order)

    V = TestFESpace(model, reffe; conformity=:H1, dirichlet_tags="boundary")
    U = TrialFESpace(V)

# 3. Define the coefficients of the elliptic equation:
    A_fn_tmp0 = Latex_A_fn.subs(fn_values).subs(param_values).doit()
    A_fn_tmp = build_Symbolics_fn(A_fn_tmp0, [x, y])
    A_fn(xx) = (typeof(xx[1]) <: Real ? A_fn_tmp([xx[1], xx[2]]) : 1.0)     # Might get called with non-Float argument

    B_fn_tmp0 = Latex_B_fn.subs(fn_values).subs(param_values).doit()
    B_fn_tmp = lambdify(B_fn_tmp0, [x, y])                      # This is where the barolinic pressure gradient term is inserted
    B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp(xx[1], xx[2]) : 0.0)     # Might get called with non-Float argument
    # This is too slow:
    # B_fn_tmp = build_Symbolics_fn(B_fn_tmp0, [x, y])          # This is where the barolinic pressure gradient term is inserted
    # B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp(xx) : 0.0)     # Might get called with non-Float argument
    
    # Testing
    println()
    println("Test comparison: A_fn(Point(0.0,0.0)) = $(A_fn(Point(0.0,0.0)))")
    println("Test comparison: B_fn(Point(0.5,0.0)) = $(B_fn(Point(0.5,0.0)))")

# 4. Define weak form (variational formulation)
    a(u,v) = ∫( real( (∇(v) ⋅ VectorValue( 1.0, -1im)) * (A_fn * (∇(u) ⋅ VectorValue(1.0, 1im))) ) )dΩ
    l(v)   = ∫( real( (∇(v) ⋅ VectorValue(-1.0,  1im)) *  B_fn ) )dΩ

# 5. Assemble and solve
    op = AffineFEOperator(a, l, U, V)
    psurf = Gridap.solve(op)  # This is your numerical solution as a Gridap FEFunction
end

# 6. Visualization with Paraview
writevtk(Ω,notebook_name * "_psurf_solution",cellfields=["psurf"=>psurf])

#### Setup helper functions:

In [ ]:
function ps_val(xx, yy)
    try
    	gp = evaluate(psurf, Point(xx, yy))
	    return gp
    catch
        return 0.0
    end
end
println("Test comparison: Value of surface pressure at the origin: $(ps_val(0.0,0.0))")

function psg_val(xx, yy)
    try
	    tmp = evaluate(∇(psurf), Point(xx, yy))
	    return tmp[1] + 1im*tmp[2] 
    catch
        return 0.0 + 1im*0.0
    end
end

function UV_fn(xx,yy) 
    try
        psg_tmp = evaluate(∇(psurf), Point(xx, yy))
        UV = A_fn_tmp([xx,yy])*(psg_tmp[1] + 1im*psg_tmp[2]) + B_fn_tmp(xx,yy)
        return UV
    catch
        return 0.0 + 1im*0.0
    end
end

### Solve for velocity field using the Green's function:

In [ ]:
println()
println("Computing velocity field from surface pressure, Green's function, and known parameters and fields:")

# Make Julia Symbolics functions from the symbolic expressions to accelerate for loop:
this_b_tmp 	     = b.subs(fn_values).subs(B_values).subs(ξ,z).subs(param_values).doit()
Guv0_fn          = build_Symbolics_fn(Guv0.subs(         param_values).subs(fn_values), [x, y, z])
Guv_int_wrt_ξ_fn = build_Symbolics_fn(sympy.simplify(Guv_int_wrt_ξ).subs(param_values).subs(fn_values).doit(), [x, y, z])
S_fn_tmp		 = build_Symbolics_fn(Latex_S_fn.subs(   param_values).subs(fn_values).rewrite(exp).doit(), [x, y])		# Need to rewrite to exp to avoid issues with SymPy's tanh being converted to Symbolics' tanh	

# T_fn_tmp		 = build_Symbolics_fn(Latex_T_fn, [x, y])				# This is too slow: use lambdify instead
T_fn_tmp		 = lambdify(Latex_T_fn, [x, y])
τb_fn(xx,yy)     = S_fn_tmp([xx,yy])*psg_val(xx,yy) + T_fn_tmp(xx,yy)
println()
println("Test comparison: S_fn_tmp(0.4,-0.1) = $(S_fn_tmp([0.4,-0.1]))")
println("Test comparison: T_fn_tmp(0.4,-0.1) = $(T_fn_tmp(0.4,-0.1))")
println("Test comparison: τb_fn(   0.4,-0.1) = $(τb_fn(0.4,-0.1))")
b_fn             = build_Symbolics_fn(this_b_tmp, [x, y, z])
println("Test comparison: Latex_b(Point(0.4,-0.0,-0.24)) = $(b_fn([0.4,-0.0,-0.24]))")

# This term can't be computed by SymPy to check. 
# The individial pieces have been checked using the Latex notes.
# It's tested numerically for a specific set of parameter values below.
pbarog_term      = ((α * γ)/(κ₀ * ψ^3 * ϵ^2)) * ((exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y))) / (exp(2 * ψ * H(x,y)) + 1)^2) * 
(sympy.diff(H(x,y),x) + im * sympy.diff(H(x,y),y)) * 
(Latex_Guv_exp_psi_xi_int_wrt_ξ - 2 * Latex_Guv_int_wrt_ξ + Latex_Guv_exp_psi_xi_int_wrt_ξ.subs(ψ,-ψ))
this_pbarog_term = pbarog_term.subs(param_values).subs(fn_values).doit()
pbarog_term_fn = lambdify(this_pbarog_term,[x,y,z])
println("Test comparison: pbarog_term_fn(0.4,-0.1,-0.14) = $(pbarog_term_fn(0.4,-0.1,-0.14))")
# This is too slow: use lambdify instead:
# pbarog_term_fn = build_Symbolics_fn(this_pbarog_term, [x, y, z])

# Nx, Ny, Nz = 129, 129, 64
# Nx, Ny, Nz = 65, 65, 32
Nx, Ny, Nz = 32, 32, 16
xs     = range(-1, 1, Nx)
ys     = range(-1, 1, Ny)
zs     = range(-1, 0, Nz)
us     = zeros(Nx, Nz)
vs     = zeros(Nx, Nz)
bs     = zeros(Nx, Nz)
Us     = zeros(Nx, Ny)
Vs     = zeros(Nx, Ny)
τxs    = zeros(Nx, Ny)
τys    = zeros(Nx, Ny)
τbxs   = zeros(Nx, Ny)
τbys   = zeros(Nx, Ny)
pss    = zeros(Nx, Ny)
spds   = zeros(Nx, Ny)

@time begin
# @profview begin
for (ix, xx) in enumerate(xs)
	for (iy, yy) in enumerate(ys)
		this_H = H_val(xx, yy)
		if this_H >= 0
			this_pss_val = ps_val( xx, yy)      # Interpolate or evaluate ps
			this_psg_val = psg_val(xx, yy)      # Interpolate or evaluate psg
			this_τs_val  = τs_val(xx, yy)		# Compute surface stress
			if iy == ceil(Int,Ny/2)				# Only compute 2D slice at y=0: for axisymmetric cases
				for (iz, zz) in enumerate(zs)
					if zz <= 0 && zz >= -this_H 
						the_psg_term	= Guv_int_wrt_ξ_fn([xx, yy, zz]) * this_psg_val 
						the_pbarog_term = pbarog_term_fn(   xx, yy, zz )					# This is a lambdify function.
						the_stress_term = Guv0_fn(         [xx, yy, zz]) * this_τs_val
						this_uv_value = the_psg_term + the_pbarog_term + the_stress_term
						us[ix, iz] = real(this_uv_value)
						vs[ix, iz] = imag(this_uv_value)
						bs[ix, iz] = b_fn([xx, yy, zz])
						if(ix == 10 && iz == 8)
							println()
							println("Test comparison at point (x,y,z) = ($(xx),$(yy),$(zz)):")
							println("Test comparison: bs[x,y,z]       = $(bs[ix, iz])")
							println("Test comparison: this_psg_val    = $(this_psg_val)")
							println("Test comparison: this_τval       = $(this_τs_val)")
							println("Test comparison: this_uv_val     = $(this_uv_value)")
							println("Test comparison: the_psg_term    = $(the_psg_term)")
							println("Test comparison: the_stress_term = $(the_stress_term)")
							println("Test comparison: the_pbarog_term = $(the_pbarog_term)")
							# Check that the_pbarog_term is accurate compared to numerical integration:
							# We want to check that integral Guv * Latex_dGb_int_int_∂z.subs(z,ξ), (ξ,-H,0) = the_pbarog_term
							# SymPy struggles to perform the integral, so customize it to specific parameters and compute numerically:
							integrand = Guv * Latex_dGb_int_int_∂z.subs(z,ξ)
							integrand = integrand.subs(param_values).subs(fn_values)
							integrand = integrand.subs(x,xx).subs(y,yy).subs(z,zz).doit()
							integrand = sympy.expand(integrand)
							integrand_fn = lambdify(integrand,[ξ])
							# integral  = sympy.integrate(integrand,(ξ,-this_H,0))			# This doesn't finish within my patience threshold...
							using QuadGK
							result, err = quadgk(integrand_fn, -this_H, 0; rtol=1e-10, atol=1e-12, maxevals=1_000_000)
							println("Test comparison: the_pbarog_term = $(result) ± $(err) from numerical quadrature")		
						end
					end
				end
			end
			# Compute 2D fields here:
			UV             = UV_fn(xx,yy)
			Us[    ix, iy] = float(real(UV))
			Vs[    ix, iy] = float(imag(UV))
			spds[  ix, iy] = sqrt(Us[ix, iy]^2 + Vs[ix, iy]^2)
			τxs[   ix, iy] = float(real(this_τs_val))
			τys[   ix, iy] = float(imag(this_τs_val))
			τb_val         = τb_fn(xx,yy)
			τbxs[  ix, iy] = float(real(τb_val))
			τbys[  ix, iy] = float(imag(τb_val))
			pss[   ix, iy] = this_pss_val
		end
	end
end
# end
end

# Write out solution for display by Paraview:
# vtk_grid(notebook_name * "_3D_solution", xs, ys, zs) do vtk
vtk_grid(notebook_name * "_3D_solution", xs, zs) do vtk
	vtk["u_speed"]  = us
	vtk["v_speed"]  = vs
	vtk["b_field"]  = bs
end

vtk_grid(notebook_name * "_2D_solution", xs, ys) do vtk
	vtk["U_speed"]      = Us
	vtk["V_speed"]      = Vs
	vtk["speed"]        = spds
	vtk["x_sfc_stress"] = τxs
	vtk["y_sfc_stress"] = τys
	vtk["x_bot_stress"] = τbxs
	vtk["y_bot_stress"] = τbys
	vtk["sfc_p"]        = pss
end